# 🛡️ Pipeline d'Ingestion & Détection de Menaces sur Logs (SIEM)
### Projet Python — Majeure Ingénieur 3 (ECE Paris)
---
**Auteur** : Équipe Projet ECE  
**Date** : Mars 2026  
**Objectif** : Démonstration interactive et rapport synthétique des 5 modules du SIEM : ingestion en streaming, programmation fonctionnelle, détection d'intrusions, moteur de règles et audit par introspection.

## 1. Initialisation de l'Environnement et Imports
Le projet repose sur une architecture modulaire située dans le package `src/`.

In [ ]:
import sys
from pathlib import Path
from collections import defaultdict

# Ajout du dossier racine au PATH Python pour importer proprement les modules de src
ROOT_DIR = Path(".").resolve().parent if Path(".").resolve().name == "Documentation" else Path(".").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.ingestion import lire_logs, lire_all_logs, parse_logs, extract_blacklist_ips
from src.transfo import filtrage_logs, normalize_logs
from src.agregation import brute_force_detector, occurence, extract_unique_error_ips, occurence_ip_connexion
from src.rules_engine import apply_security_rules
from src.introspection import introspection_local, introspection_global, list_info, formate_affichages

print("✅ Tous les modules ont été importés avec succès !")

## 2. Module 1 : Ingestion en Streaming et Parsing JSON
Vérification du fonctionnement des générateurs paresseux (`yield`) et du découpage avec désérialisation JSON.

In [ ]:
# Aperçu des 3 premières lignes ingérées de manière paresseuse
log_sample_path = str(ROOT_DIR / "demo_data" / "auth.log")
generateur_brut = lire_logs(log_sample_path)

print("--- Échantillon de lecture ligne par ligne (Module 1) ---")
for _ in range(3):
    ligne_texte, num_ligne, chemin = next(generateur_brut)
    print(f"[Ligne {num_ligne}] Fichier: {Path(chemin).name}")
    print(f"Contenu brut : {ligne_texte}")
    
    # Parsing en dictionnaire structuré
    parsed = parse_logs((ligne_texte, num_ligne, chemin))
    print(f"Objet parsé  : {parsed}\n")

## 3. Module 2 : Filtrage et Normalisation par Pattern Matching
Démonstration de `match...case` sur différents types de données (chaînes, dictionnaires, tuples).

In [ ]:
# Exemple d'objet composite non normalisé
test_log = {
    "time": "2026-03-11 12:00:00",
    "level": "warning",
    "ip": "192.168.1.105",
    "user": "admin_svc",
    "payload": {
        "event": "login_failed",
        "port": 22,
        "protocol": "ssh"
    },
    "metadata": (22, "ssh", None)
}

normalise = normalize_logs(test_log)
print("Log normalisé par Pattern Matching (Module 2) :\n", normalise)

## 4. Module 3 : Exécution du Pipeline Complet et Détection de Menaces
Exécution du flux complet sur l'ensemble des journaux de `demo_data/`.

In [ ]:
context = {
    'occurence_brute_force': defaultdict(int),
    'ip_sus_brute_force': set(),
    'occurence_user': defaultdict(int),
    'occurence_error': defaultdict(int),
    'unique_ips': set(),
    'echecs_par_ip': defaultdict(int),
    'success_par_ip': defaultdict(int),
    'ports': defaultdict(int),
    'protocols': defaultdict(int),
    'status_codes': defaultdict(int)
}

logs = [
    str(ROOT_DIR / 'demo_data' / 'auth.log'),
    str(ROOT_DIR / 'demo_data' / 'network.log'),
    str(ROOT_DIR / 'demo_data' / 'server.log'),
]
blacklist_path = str(ROOT_DIR / 'demo_data' / 'blacklist.txt')

# Traitement en streaming complet
raw_logs = lire_all_logs(logs)
parsed_logs = map(parse_logs, raw_logs)
filtered_logs = filtrage_logs(parsed_logs)
normalized_logs = map(normalize_logs, filtered_logs)
blacklist_ips = extract_blacklist_ips(blacklist_path)

compteur = 0
for line_log in normalized_logs:
    compteur += 1
    var_locals = apply_security_rules(
        brute_force_detector,
        occurence,
        extract_unique_error_ips,
        occurence_ip_connexion,
        log=line_log,
        context=context
    )
    introspection_local(var_locals)

print(f"✅ {compteur} événements de logs traités avec succès !")
print(f"🚨 Adresses IP suspectées d'attaque par force brute : {context['ip_sus_brute_force']}")
print(f"⚠️ Répartition des erreurs par niveau : {dict(context['occurence_error'])}")

## 5. Corrélation Ensembliste avec la Blacklist
Application de l'algèbre ensembliste pour classifier les menaces :

In [ ]:
actives = context['unique_ips']
suspectes = context['ip_sus_brute_force']

print("=== ANALYSE ENSEMBLISTE DES IP ===")
print(f"• IP actives avec anomalies       : {len(actives)}")
print(f"• IP dans la Blacklist            : {len(blacklist_ips)}")
print(f"• IP Blacklistées et Actives (&)  : {actives & blacklist_ips}")
print(f"• Nouvelles menaces hors liste (-) : {suspectes - blacklist_ips}")
print(f"• Différence symétrique (^)       : {len(actives ^ blacklist_ips)} adresses")

## 6. Synthèse Graphique des Indicateurs de Sécurité
Visualisation synthétique des résultats d'audit.

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Répartition des niveaux d'anomalies
    niveaux = list(context['occurence_error'].keys())
    valeurs_niveaux = list(context['occurence_error'].values())
    ax1.bar(niveaux, valeurs_niveaux, color=['#e74c3c', '#f39c12', '#c0392b'])
    ax1.set_title("Distribution des Niveaux de Sévérité Anormaux")
    ax1.set_ylabel("Nombre d'occurrences")
    
    # 2. Protocoles réseau les plus actifs
    proto = list(context['protocols'].keys())
    val_proto = list(context['protocols'].values())
    ax2.pie(val_proto, labels=proto, autopct='%1.1f%%', startangle=140, colors=['#3498db', '#2ecc71', '#9b59b6', '#34495e'])
    ax2.set_title("Répartition des Protocoles Réseau")
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print("ℹ️ matplotlib n'est pas installé. Exécutez 'pip install matplotlib' pour afficher les graphiques.")

## 7. Clôture d'Audit et Génération des Rapports
Appel d'introspection globale pour sauvegarder les indicateurs consolidés dans `report/report_global.txt`.

In [ ]:
var_globals = globals()
introspection_global(var_globals)
print("📁 Rapports générés avec succès :")
print("  - report/report_local.txt  (Alertes au fil de l'eau)")
print("  - report/report_global.txt (Synthèse statistique globale)")